# Exploratory Analysis — Closet Monitor

**Question: What does the environmental data from the network closet look like, and what patterns exist?**

This notebook covers:
1. Data loading and summary statistics
2. Sampling cadence verification (is the sensor reliable?)
3. Temperature and humidity time-series visualization
4. Temperature-humidity correlation analysis
5. HVAC cycle detection using smoothing + peak finding
6. Cycle length analysis and time-of-day patterns

For anomaly detection (identifying unusual events), see [02-anomaly-detection.ipynb](02-anomaly-detection.ipynb).

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Pretty defaults
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["figure.dpi"] = 100

# Load data from SQLite into a pandas DataFrame
db_path = Path("../data/closet.db")
conn = sqlite3.connect(db_path)
df = pd.read_sql_query(
    "SELECT * FROM readings ORDER BY received_at",
    conn,
    parse_dates=["received_at"],
)
conn.close()

# Convert UTC timestamps to local time (Central) so charts make intuitive sense
df["received_at"] = df["received_at"].dt.tz_convert("America/Chicago")
df = df.set_index("received_at")

print(f"Loaded {len(df):,} readings")
print(f"Time range: {df.index.min()} → {df.index.max()}")
print(f"Duration: {df.index.max() - df.index.min()}")
df.head()

In [ ]:
# Quick summary of every numeric column
df[["temp_f", "humidity", "pressure_hpa", "rssi"]].describe().round(2)

In [ ]:
# How spaced out are consecutive readings?
intervals = df.index.to_series().diff().dt.total_seconds().dropna()
print(f"Median interval: {intervals.median():.1f}s")
print(f"Mean interval:   {intervals.mean():.1f}s")
print(f"Max interval:    {intervals.max():.1f}s  ← any big gap means a dropout")
print(f"Min interval:    {intervals.min():.1f}s")
print(f"\nReadings with interval > 60s (= dropout): {(intervals > 60).sum()}")

In [ ]:
fig, ax = plt.subplots()
df["temp_f"].plot(ax=ax, color="#d62728", linewidth=1.2)
ax.set_title("Closet Temperature Over Time", fontsize=14, fontweight="bold")
ax.set_ylabel("Temperature (°F)")
ax.set_xlabel("")
ax.axhline(y=85, color="red", linestyle="--", alpha=0.4, label="Alert threshold (85°F)")
ax.axhline(y=df["temp_f"].mean(), color="gray", linestyle=":", alpha=0.5, label=f"Mean ({df['temp_f'].mean():.1f}°F)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots()
df["humidity"].plot(ax=ax, color="#1f77b4", linewidth=1.2)
ax.set_title("Closet Humidity Over Time", fontsize=14, fontweight="bold")
ax.set_ylabel("Relative Humidity (%)")
ax.set_xlabel("")
ax.axhline(y=60, color="red", linestyle="--", alpha=0.4, label="High alert (60%)")
ax.axhline(y=25, color="orange", linestyle="--", alpha=0.4, label="Low alert (25%)")
ax.axhline(y=df["humidity"].mean(), color="gray", linestyle=":", alpha=0.5, label=f"Mean ({df['humidity'].mean():.1f}%)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, ax1 = plt.subplots()

ax1.plot(df.index, df["temp_f"], color="#d62728", linewidth=1.2, label="Temperature")
ax1.set_ylabel("Temperature (°F)", color="#d62728")
ax1.tick_params(axis="y", labelcolor="#d62728")

ax2 = ax1.twinx()
ax2.plot(df.index, df["humidity"], color="#1f77b4", linewidth=1.2, label="Humidity")
ax2.set_ylabel("Humidity (%)", color="#1f77b4")
ax2.tick_params(axis="y", labelcolor="#1f77b4")

plt.title("Temperature vs Humidity Over Time", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
correlation = df["temp_f"].corr(df["humidity"])
print(f"Pearson correlation between temp and humidity: {correlation:.3f}")
print()
if correlation < -0.5:
    print("→ Strong negative correlation: classic HVAC behavior (warm air, drier readings)")
elif correlation > 0.5:
    print("→ Strong positive correlation: heat + moisture together (human presence?)")
else:
    print("→ Weak correlation: temperature and humidity vary somewhat independently")

In [ ]:
# === Understanding pandas, one piece at a time ===

# 1. df is a DataFrame — think of it as a SQL table or Excel spreadsheet in memory
print("Type of df:", type(df).__name__)
print("Shape (rows, columns):", df.shape)
print()

# 2. Each column is a Series — a 1D array with the timestamp as its label
print("Type of df['temp_f']:", type(df["temp_f"]).__name__)
print("First 3 values:")
print(df["temp_f"].head(3))
print()

# 3. The index is what makes this a "time-series" DataFrame
print("Index type:", type(df.index).__name__)
print("This is special — pandas knows these are timestamps, so we get free time math")
print()

# 4. Time math example: get just the readings between 6 AM and 7 AM today
morning_window = df.between_time("06:00", "07:00")
print(f"Readings between 6-7 AM: {len(morning_window)}")
print(f"Avg temp during that window: {morning_window['temp_f'].mean():.2f}°F")
print(f"Avg humidity during that window: {morning_window['humidity'].mean():.2f}%")

In [ ]:
# Zoom in on what happened around 06:00 on April 15
spike_window = df.loc["2026-04-15 05:30":"2026-04-15 07:00"]

fig, ax1 = plt.subplots(figsize=(14, 5))
ax1.plot(spike_window.index, spike_window["temp_f"], color="#d62728", marker="o", markersize=3, label="Temp")
ax1.set_ylabel("Temperature (°F)", color="#d62728")
ax1.tick_params(axis="y", labelcolor="#d62728")

ax2 = ax1.twinx()
ax2.plot(spike_window.index, spike_window["humidity"], color="#1f77b4", marker="o", markersize=3, label="Humidity")
ax2.set_ylabel("Humidity (%)", color="#1f77b4")
ax2.tick_params(axis="y", labelcolor="#1f77b4")

plt.title("The 6 AM Event — Zoomed In", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# What were the actual numbers?
print("\nReading-by-reading during the event:")
print(spike_window[["temp_f", "humidity", "rssi"]].head(20).round(2))

In [ ]:
# Apply a rolling mean to smooth out sensor jitter
# Window of 5 readings = 2.5 minutes of averaging
df["temp_smooth"] = df["temp_f"].rolling(window=5, center=True).mean()

# Plot raw vs smoothed so you can see what smoothing does
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df.index, df["temp_f"], color="lightcoral", linewidth=0.8, alpha=0.5, label="Raw")
ax.plot(df.index, df["temp_smooth"], color="darkred", linewidth=1.5, label="Smoothed (5-pt rolling mean)")
ax.set_title("Raw vs Smoothed Temperature", fontsize=14, fontweight="bold")
ax.set_ylabel("Temperature (°F)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
from scipy.signal import find_peaks

# We need scipy for peak detection — install it if you don't have it
# (run this in a terminal if find_peaks errors: pip install scipy)

# Drop NaN values from the smoothed series (rolling window creates NaN at edges)
temp_clean = df["temp_smooth"].dropna()

# Find peaks (local maxima) — these are when AC kicks ON (temp stops rising)
# distance=10 means peaks must be at least 10 readings (5 minutes) apart
# prominence=0.3 means a peak must rise at least 0.3°F above its surroundings
peaks_idx, peak_props = find_peaks(temp_clean.values, distance=10, prominence=0.3)

# Find troughs (local minima) — these are when AC turns OFF (temp stops falling)
# We invert the signal to find minima as if they were maxima
troughs_idx, trough_props = find_peaks(-temp_clean.values, distance=10, prominence=0.3)

peaks = temp_clean.iloc[peaks_idx]
troughs = temp_clean.iloc[troughs_idx]

print(f"Detected {len(peaks)} peaks (AC kick-on events)")
print(f"Detected {len(troughs)} troughs (AC turn-off events)")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(temp_clean.index, temp_clean.values, color="darkred", linewidth=1.2, label="Smoothed temp")
ax.scatter(peaks.index, peaks.values, color="red", s=80, marker="v", zorder=5, label=f"Peaks ({len(peaks)})")
ax.scatter(troughs.index, troughs.values, color="blue", s=80, marker="^", zorder=5, label=f"Troughs ({len(troughs)})")
ax.set_title("Detected HVAC Cycles", fontsize=14, fontweight="bold")
ax.set_ylabel("Temperature (°F)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Time between consecutive peaks = full HVAC cycle length
peak_times = peaks.index.to_series()
cycle_lengths = peak_times.diff().dropna().dt.total_seconds() / 60  # in minutes

print("=== HVAC Cycle Statistics ===")
print(f"Total cycles detected:       {len(peaks)}")
print(f"Average cycle length:        {cycle_lengths.mean():.1f} minutes")
print(f"Median cycle length:         {cycle_lengths.median():.1f} minutes")
print(f"Shortest cycle:              {cycle_lengths.min():.1f} minutes")
print(f"Longest cycle:               {cycle_lengths.max():.1f} minutes")
print(f"Std deviation:               {cycle_lengths.std():.1f} minutes")
print()

# Temperature swing per cycle = peak temp - following trough temp
print("=== Temperature Swing ===")
print(f"Average peak temperature:    {peaks.mean():.2f}°F")
print(f"Average trough temperature:  {troughs.mean():.2f}°F")
print(f"Average temperature swing:   {peaks.mean() - troughs.mean():.2f}°F per cycle")

In [ ]:
# Distribution of cycle lengths — does the data confirm the mean/median story?
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

# Histogram of cycle lengths
ax1.hist(cycle_lengths, bins=10, color="steelblue", edgecolor="white")
ax1.axvline(cycle_lengths.mean(), color="red", linestyle="--", label=f"Mean: {cycle_lengths.mean():.1f} min")
ax1.axvline(cycle_lengths.median(), color="orange", linestyle="--", label=f"Median: {cycle_lengths.median():.1f} min")
ax1.set_title("Distribution of HVAC Cycle Lengths", fontweight="bold")
ax1.set_xlabel("Cycle length (minutes)")
ax1.set_ylabel("Number of cycles")
ax1.legend()

# Cycle length over time — are long cycles clustered at any time of day?
ax2.scatter(peak_times.iloc[1:], cycle_lengths, color="darkred", s=60)
ax2.set_title("Cycle Length by Time of Day", fontweight="bold")
ax2.set_ylabel("Cycle length (minutes)")
ax2.set_xlabel("Time of cycle start")
ax2.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

# Show the actual cycle lengths in order
print("\nAll detected cycle lengths (in chronological order):")
for i, (start_time, length) in enumerate(zip(peak_times.iloc[1:], cycle_lengths), 1):
    flag = "  ← LONG" if length > 90 else ("  ← short" if length < 25 else "")
    print(f"  Cycle {i:2d}: {start_time.strftime('%H:%M')}  →  {length:5.1f} min{flag}")

## Key findings

1. **Sensor reliability:** Zero packet loss across the entire observation window. Median interval exactly 30.0s.
2. **Temperature range:** Tight 5.4°F band (71.8–77.2°F) with std dev of 0.91°F — healthy HVAC control.
3. **Humidity range:** 42.4–57.5%, averaging 46.9%. Brief spikes detected but no sustained excursions.
4. **Correlation:** Weak (-0.099) between temperature and humidity — multiple independent variables are influencing the environment, not just HVAC.
5. **HVAC cycles:** 15 cycles detected automatically. Two distinct regimes: short cycles (16–50 min) during evening peak hours, long cycles (100+ min) overnight. Average swing of 1.29°F per cycle.
6. **Longest cycle (177 min)** occurred in the early morning when outside temperatures were lowest — consistent with healthy demand-driven HVAC behavior.

**Next:** See [02-anomaly-detection.ipynb](02-anomaly-detection.ipynb) for rate-of-change anomaly detection and event identification.